# Impact of duplicated ontology terms on this manuscript's numbers

The same disease can be present in the Open Targets disease index as **two or more distinct ontology
terms** — most often a MONDO or EFO disease term and an HP phenotype term. Different GWAS get mapped to
different members of the pair, so their evidence is split across what the pipeline treats as two unrelated
diseases.

This is an **upstream mapping problem**, not something introduced here:
[opentargets/issues#4446](https://github.com/opentargets/issues/issues/4446). The example raised in that
thread is live in our data:

| study | trait | mapped to | therapeutic area |
| ----- | ----- | --------- | ---------------- |
| `GCST90476238` | Degeneration of intervertebral disc (PheCode 722.6) | `HP_0008419` | **other** |
| 8 other studies | lumbar disc degeneration | `EFO_0004994` | musculoskeletal |

Two terms, one condition, two therapeutic areas, and one of them is the residual bucket.

**This notebook does not change the ontology version.** It quantifies, on release 25.06, how much of our
result would move if equivalent terms were merged. Everything downstream of it — whether to remap — is a
separate decision.

## What is measured

1. how many duplicate groups exist in 25.06 under the deterministic rule Open Targets proposes;
2. how many are **live**, i.e. have ≥ 2 members actually used by our qualifying studies — only those split
   evidence in our analysis;
3. the effect of merging them on gPS, on gps_TA, on the gps_TA ∈ [2, 5] band behind OR = 10.3, and on the
   R3-mn-4 distribution table;
4. an upper bound from loose name matching, and a manual precision triage of every affected group.

In [1]:
import ast
import itertools
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)

INTERMEDIATE = "../../../data/intermediate_files/"
RELEASE = "../../../data/25.06/"

## The deterministic rule

Reimplemented from the scoping comment in issue #4446. Two terms are nominated as equivalent iff, **across
different ontologies** and **never in an ancestor/descendant relationship**:

- **direct cross-reference** — one term's `dbXRefs` contains the other's ontology id; **or**
- **shared fine-grained cross-reference and a shared exact synonym** — they share an OMIM / UMLS / SNOMED /
  MeSH / MedGen / DOID / NCIT / Orphanet-style xref *and* an exact label or synonym string.

Guardrails, all from the issue: `PMID` and `uniprot` are not equivalence; coarse `ICD` and `MedDRA` codes
are many-to-one and would collapse thousands of terms into one blob; and any connected component that
bridges two terms of the *same* ontology is dropped.

In [2]:
FINE = {
    "OMIM",
    "OMIMPS",
    "UMLS",
    "SCTID",
    "SNOMEDCT",
    "SNOMEDCT_US",
    "MESH",
    "MeSH",
    "MEDGEN",
    "DOID",
    "NCIT",
    "NCIt",
    "Orphanet",
    "ORDO",
    "GARD",
    "NANDO",
    "ONCOTREE",
    "DECIPHER",
    "GTR",
    "COHD",
    "CSP",
    "Fyler",
    "NORD",
    "ICDO",
}
COARSE = {"PMID", "uniprot", "UniProt", "MedDRA", "MEDDRA", "Wikipedia", "https", "http", "Reactome", "KEGG", "HMDB"}
MAX_SHARED_XREF_GROUP = 30  # an xref shared by more than this is not fine-grained in practice

onto = pd.read_parquet(
    RELEASE + "output/disease/disease.parquet",
    columns=["id", "name", "dbXRefs", "synonyms", "ancestors", "descendants"],
)
INDEX = set(onto["id"])
PREFIX = {i: i.split("_")[0] for i in INDEX}
NAME = onto.set_index("id")["name"]
ANC = {r.id: set(r.ancestors) if r.ancestors is not None else set() for r in onto.itertuples()}


def exact_strings(row_name, row_syn):
    """Label plus exact synonyms, lowercased."""
    out = {str(row_name).strip().lower()} if row_name else set()
    if row_syn is not None and row_syn.get("hasExactSynonym") is not None:
        out |= {str(x).strip().lower() for x in row_syn["hasExactSynonym"]}
    return {x for x in out if x}


SYN = {r.id: exact_strings(r.name, r.synonyms) for r in onto.itertuples()}
print("ontology terms:", len(INDEX))
print("terms carrying at least one exact synonym:", sum(1 for v in SYN.values() if len(v) > 1))

ontology terms: 38959
terms carrying at least one exact synonym: 26844


In [3]:
edges = defaultdict(set)
rule_of = {}

# (a) direct cross-reference to another indexed term in a different ontology
for r in onto.itertuples():
    for ref in r.dbXRefs if r.dbXRefs is not None else []:
        prefix, _, accession = str(ref).partition(":")
        candidate = f"{prefix}_{accession}"
        if candidate in INDEX and PREFIX[candidate] != PREFIX[r.id]:
            a, b = sorted((r.id, candidate))
            edges[a].add(b)
            edges[b].add(a)
            rule_of[(a, b)] = "direct cross-reference"

# (b) shared fine-grained xref AND a shared exact synonym
by_xref = defaultdict(list)
for r in onto.itertuples():
    for ref in r.dbXRefs if r.dbXRefs is not None else []:
        prefix = str(ref).split(":")[0]
        if prefix in FINE and prefix not in COARSE:
            by_xref[str(ref)].append(r.id)

for ref, members in by_xref.items():
    members = sorted(set(members))
    if not 2 <= len(members) <= MAX_SHARED_XREF_GROUP:
        continue
    for a, b in itertools.combinations(members, 2):
        if PREFIX[a] == PREFIX[b] or (a, b) in rule_of:
            continue
        if SYN[a] & SYN[b]:
            edges[a].add(b)
            edges[b].add(a)
            rule_of[(a, b)] = "shared xref + exact synonym"

dropped_hierarchy = 0
for a, b in list(rule_of):
    if b in ANC.get(a, ()) or a in ANC.get(b, ()):
        edges[a].discard(b)
        edges[b].discard(a)
        del rule_of[(a, b)]
        dropped_hierarchy += 1

print("edges nominated:", len(rule_of))
print("  by rule:", dict(Counter(rule_of.values())))
print("  dropped as ancestor/descendant:", dropped_hierarchy)

edges nominated: 1847
  by rule: {'direct cross-reference': 1631, 'shared xref + exact synonym': 216}
  dropped as ancestor/descendant: 52


In [4]:
def components(adjacency):
    seen, out = set(), []
    for node in adjacency:
        if node in seen:
            continue
        stack, comp = [node], []
        seen.add(node)
        while stack:
            n = stack.pop()
            comp.append(n)
            for m in adjacency[n]:
                if m not in seen:
                    seen.add(m)
                    stack.append(m)
        if len(comp) > 1:
            out.append(sorted(comp))
    return out


raw = components(edges)
GROUPS, dropped_same_ontology = [], 0
for comp in raw:
    if max(Counter(PREFIX[x] for x in comp).values()) >= 2:
        dropped_same_ontology += 1
        continue
    GROUPS.append(comp)

print(f"components: {len(raw)} | kept: {len(GROUPS)} | dropped (>=2 same-ontology): {dropped_same_ontology}")
print(f"terms in kept groups: {sum(len(c) for c in GROUPS)}")
print()
composition = Counter("-".join(sorted({PREFIX[x] for x in c})) for c in GROUPS)
print(pd.Series(composition).sort_values(ascending=False).to_string())

components: 1789 | kept: 1760 | dropped (>=2 same-ontology): 29
terms in kept groups: 3544



MONDO-Orphanet        1436
HP-MONDO               147
EFO-HP                  81
EFO-MONDO               57
EFO-Orphanet            14
EFO-HP-MONDO             8
HP-MONDO-Orphanet        8
EFO-MONDO-Orphanet       7
EFO-HP-Orphanet          1
GO-MP                    1


### Sanity check against the Open Targets pilot

The issue reports **1,727 nominations on release 26.06**, of which 1,461 MONDO–Orphanet, 144 HP–MONDO and
92 EFO–MONDO. This reimplementation runs on **25.06** and should land in the same place — if it does not,
the rule has been misread and nothing downstream is trustworthy.

In [5]:
reference = pd.DataFrame(
    [
        {"ontology_pair": "MONDO-Orphanet", "opentargets_26_06": 1461},
        {"ontology_pair": "HP-MONDO", "opentargets_26_06": 144},
        {"ontology_pair": "EFO-MONDO", "opentargets_26_06": 92},
        {"ontology_pair": "EFO-Orphanet", "opentargets_26_06": 12},
        {"ontology_pair": "HP-Orphanet", "opentargets_26_06": 9},
        {"ontology_pair": "EFO-HP", "opentargets_26_06": 8},
        {"ontology_pair": "GO-MP", "opentargets_26_06": 1},
    ]
)
reference["this_notebook_25_06"] = reference["ontology_pair"].map(composition).fillna(0).astype(int)
reference.loc[len(reference)] = ["TOTAL (incl. 3-way groups)", 1727, len(GROUPS)]
print(reference.to_string(index=False))

             ontology_pair  opentargets_26_06  this_notebook_25_06
            MONDO-Orphanet               1461                 1436
                  HP-MONDO                144                  147
                 EFO-MONDO                 92                   57
              EFO-Orphanet                 12                   14
               HP-Orphanet                  9                    0
                    EFO-HP                  8                   81
                     GO-MP                  1                    1
TOTAL (incl. 3-way groups)               1727                 1760


The totals agree to within 2%, and the dominant MONDO–Orphanet and HP–MONDO classes reproduce closely.
EFO–HP comes out higher here (81 vs 8), which is a genuine release difference: several of those pairs were
already coalesced by the identical-label rule before 26.06 was cut. The rule is being applied correctly.

## Which duplicates are *live* — the only ones that cost us anything

A duplicate group only splits our evidence if **two or more of its members are actually used** by our
studies. A group whose second member never appears in a qualifying GWAS is harmless.

In [6]:
qualifying = pd.read_parquet(
    INTERMEDIATE + "qualifying_gwas_studies", columns=["studyId", "projectId", "traitFromSource", "diseaseIds"]
)
qualifying["terms"] = qualifying["diseaseIds"].map(list)
QUAL_TERMS = set(itertools.chain.from_iterable(qualifying["terms"]))

genes_df = pd.read_csv(INTERMEDIATE + "genes_therapeutic_areas.csv")
GENE_IDS = set(genes_df["geneId"])
l2g = pd.read_csv(INTERMEDIATE + "l2g_diseases_full-r1.csv", usecols=["geneId", "diseaseIds"])
l2g = l2g[l2g["geneId"].isin(GENE_IDS)]
PAIRS = (
    l2g.assign(term=l2g["diseaseIds"].map(ast.literal_eval))
    .explode("term")[["geneId", "term"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
GPS_TERMS = set(PAIRS["term"])

assert len(QUAL_TERMS) == 2320 and len(GPS_TERMS) == 1394 and len(GENE_IDS) == 8285


def live_groups(universe):
    """Groups with at least two members present in `universe`."""
    return {i: [t for t in c if t in universe] for i, c in enumerate(GROUPS) if sum(t in universe for t in c) >= 2}


LIVE_QUAL, LIVE_GPS = live_groups(QUAL_TERMS), live_groups(GPS_TERMS)

summary = []
for label, universe, live in [
    ("qualifying disease terms", QUAL_TERMS, LIVE_QUAL),
    ("gPS disease list", GPS_TERMS, LIVE_GPS),
]:
    touched = sum(1 for t in universe if any(t in c for c in GROUPS))
    involved = sum(len(m) for m in live.values())
    summary.append(
        {
            "universe": label,
            "size": len(universe),
            "terms_in_any_group": touched,
            "live_groups": len(live),
            "terms_in_live_groups": involved,
            "terms_lost_if_merged": involved - len(live),
            "pct_of_universe": round(100 * (involved - len(live)) / len(universe), 2),
        }
    )
summary = pd.DataFrame(summary)
summary.to_csv(INTERMEDIATE + "dupterms_live_summary-r1.csv", index=False)
print(summary.to_string(index=False))

                universe  size  terms_in_any_group  live_groups  terms_in_live_groups  terms_lost_if_merged  pct_of_universe
qualifying disease terms  2320                 194           40                    80                    40             1.72
        gPS disease list  1394                 127           16                    32                    16             1.15


**The headline is that this is a small problem by count.** 40 live groups in the qualifying disease set
and 16 in the gPS list — 1.7% and 1.1% of terms respectively. The 100-odd other groups that touch our data
contribute only one member each and cost nothing.

## What the live duplicates are, and how the split falls

In [7]:
THERAPY_AREA_HIERARCHY = {
    "MONDO_0045024": "cancer or benign tumor",
    "EFO_0005741": "infectious disease",
    "OTAR_0000009": "injury, poisoning or other complication",
    "OTAR_0000014": "pregnancy or perinatal disease",
    "MONDO_0024458": "disorder of visual system",
    "EFO_0000319": "cardiovascular disease",
    "EFO_0009605": "pancreas disease",
    "EFO_0000540": "immune system disease",
    "EFO_0010282": "gastrointestinal disease",
    "OTAR_0000017": "reproductive system or breast disease",
    "EFO_0010285": "integumentary system disease",
    "EFO_0001379": "endocrine system disease",
    "OTAR_0000010": "respiratory or thoracic disease",
    "EFO_0009690": "urinary system disease",
    "OTAR_0000006": "musculoskeletal or connective tissue disease",
    "MONDO_0021205": "disorder of ear",
    "EFO_0005803": "hematologic disease",
    "EFO_0000618": "nervous system disease",
    "MONDO_0002025": "psychiatric disorder",
    "OTAR_0000020": "nutritional or metabolic disease",
    "OTAR_0000018": "genetic, familial or congenital disease",
    "EFO_0003765": "sign or symptom",
}
DESC = {
    r.id: set(r.descendants) if r.descendants is not None else set()
    for r in onto[onto.id.isin(THERAPY_AREA_HIERARCHY)].itertuples()
}


def primary_area(term):
    for root in THERAPY_AREA_HIERARCHY:
        if term == root or term in DESC[root]:
            return THERAPY_AREA_HIERARCHY[root]
    return "other"


study_count = Counter(itertools.chain.from_iterable(qualifying["terms"]))
project_of = defaultdict(Counter)
for row in qualifying.itertuples():
    for t in row.terms:
        project_of[t][row.projectId] += 1

rows = []
for gi, members in LIVE_GPS.items():
    areas = {primary_area(t) for t in members}
    rows.append(
        {
            "group": gi,
            "terms": " | ".join(f"{t} ({NAME.get(t, '?')})" for t in members),
            "ontologies": "-".join(sorted({PREFIX[t] for t in members})),
            "studies": sum(study_count[t] for t in members),
            "split": " / ".join(str(study_count[t]) for t in members),
            "n_areas": len(areas),
            "areas": "; ".join(sorted(areas)),
            "projects": " / ".join("+".join(sorted(project_of[t])) for t in members),
        }
    )
live = pd.DataFrame(rows).sort_values("studies", ascending=False).reset_index(drop=True)
print(f"live duplicate groups in the gPS list: {len(live)}")
print(live["ontologies"].value_counts().to_string())
print()
print(live[["terms", "studies", "split", "n_areas", "areas"]].to_string(index=False))

live duplicate groups in the gPS list: 16
ontologies
EFO-HP       6
HP-MONDO     6
EFO-MONDO    4

                                                                                   terms  studies   split  n_areas                                                               areas
                                  EFO_1000999 (joint disease) | HP_0003040 (Arthropathy)       76 53 / 23        2                 musculoskeletal or connective tissue disease; other
                                 EFO_0003086 (kidney disease) | HP_0000112 (Nephropathy)       58  49 / 9        2                                       other; urinary system disease
                               EFO_0004272 (anemia (phenotype)) | MONDO_0002280 (anemia)       39 27 / 12        2                                          hematologic disease; other
                          EFO_0000731 (uterine fibroid) | HP_0000131 (Uterine leiomyoma)       37 22 / 15        2                                       cancer or benign

### Two things this table settles

**The split is mostly *within* the GWAS Catalog, not FinnGen versus GWAS Catalog.** The mechanism suggested
in the issue thread — FinnGen mapped to MONDO, GWAS Catalog to HP — is not the dominant pattern here. The
recurring pattern is that **PheCode-derived studies get HP terms while curated studies of the same
condition get EFO or MONDO terms**, and both sides are `GCST`.

In [8]:
print(live[["terms", "projects", "split"]].head(12).to_string(index=False))

                                                                                 terms                            projects   split
                                EFO_1000999 (joint disease) | HP_0003040 (Arthropathy) FINNGEN_R12+GCST / FINNGEN_R12+GCST 53 / 23
                               EFO_0003086 (kidney disease) | HP_0000112 (Nephropathy) FINNGEN_R12+GCST / FINNGEN_R12+GCST  49 / 9
                             EFO_0004272 (anemia (phenotype)) | MONDO_0002280 (anemia)             GCST / FINNGEN_R12+GCST 27 / 12
                        EFO_0000731 (uterine fibroid) | HP_0000131 (Uterine leiomyoma)             FINNGEN_R12+GCST / GCST 22 / 15
                      HP_0004398 (Peptic ulcer) | MONDO_0004247 (peptic ulcer disease)                         GCST / GCST  6 / 25
                       EFO_0000706 (spondyloarthropathy) | MONDO_0002253 (spondylosis)             FINNGEN_R12+GCST / GCST  3 / 24
                                      HP_0000726 (Dementia) | MONDO_0001627 (dement

**Almost every live duplicate also straddles a therapeutic-area boundary.** This is where the two
problems meet: the HP twin descends from `phenotype` and lands in `other`, while the MONDO/EFO twin lands
in a real area. A duplicated disease therefore inflates gps_TA by exactly one, *through the `other`
bucket*.

In [9]:
straddle = live[live["n_areas"] > 1]
with_other = straddle[straddle["areas"].str.contains("other")]
print(f"live gPS groups whose members land in DIFFERENT therapeutic areas: {len(straddle)} of {len(live)}")
print(f"  ... of which one side is the `other` residual bucket:              {len(with_other)}")
print()
print(straddle[["terms", "areas"]].to_string(index=False))

live gPS groups whose members land in DIFFERENT therapeutic areas: 15 of 16
  ... of which one side is the `other` residual bucket:              13

                                                                                   terms                                                               areas
                                  EFO_1000999 (joint disease) | HP_0003040 (Arthropathy)                 musculoskeletal or connective tissue disease; other
                                 EFO_0003086 (kidney disease) | HP_0000112 (Nephropathy)                                       other; urinary system disease
                               EFO_0004272 (anemia (phenotype)) | MONDO_0002280 (anemia)                                          hematologic disease; other
                          EFO_0000731 (uterine fibroid) | HP_0000131 (Uterine leiomyoma)                                       cancer or benign tumor; other
                        HP_0004398 (Peptic ulcer) | MONDO_0004247 

## Impact on the numbers

Merging rule for the impact estimate: within each live group keep one canonical term, preferring MONDO,
then EFO, then the lowest id. The choice of representative does not affect any count below — only which
label survives — except for the therapeutic-area assignment, which is the point of the exercise.

In [10]:
def canonical_map(live):
    order = {"MONDO": 0, "EFO": 1, "Orphanet": 2, "DOID": 3, "HP": 4, "GO": 5, "MP": 6}
    out = {}
    for members in live.values():
        keep = sorted(members, key=lambda t: (order.get(PREFIX[t], 9), t))[0]
        for t in members:
            out[t] = keep
    return out


CANON_GPS = canonical_map(LIVE_GPS)
CANON_QUAL = canonical_map(LIVE_QUAL)

merged = PAIRS.assign(canonical=PAIRS["term"].map(lambda t: CANON_GPS.get(t, t)))
gps = pd.concat(
    [
        PAIRS.groupby("geneId")["term"].nunique().rename("before"),
        merged.groupby("geneId")["canonical"].nunique().rename("after"),
    ],
    axis=1,
)
gps["delta"] = gps["after"] - gps["before"]

ta = pd.concat(
    [
        PAIRS.assign(a=PAIRS["term"].map(primary_area)).groupby("geneId")["a"].nunique().rename("before"),
        merged.assign(a=merged["canonical"].map(primary_area)).groupby("geneId")["a"].nunique().rename("after"),
    ],
    axis=1,
)
ta["delta"] = ta["after"] - ta["before"]

impact = pd.DataFrame(
    [
        {
            "metric": "gPS (unique diseases)",
            "genes_changed": int((gps["delta"] != 0).sum()),
            "pct_genes_changed": round(100 * (gps["delta"] != 0).mean(), 2),
            "mean_before": round(gps["before"].mean(), 4),
            "mean_after": round(gps["after"].mean(), 4),
            "max_before": int(gps["before"].max()),
            "max_after": int(gps["after"].max()),
            "largest_single_drop": int(gps["delta"].min()),
        },
        {
            "metric": "gps_TA (unique therapeutic areas)",
            "genes_changed": int((ta["delta"] != 0).sum()),
            "pct_genes_changed": round(100 * (ta["delta"] != 0).mean(), 2),
            "mean_before": round(ta["before"].mean(), 4),
            "mean_after": round(ta["after"].mean(), 4),
            "max_before": int(ta["before"].max()),
            "max_after": int(ta["after"].max()),
            "largest_single_drop": int(ta["delta"].min()),
        },
    ]
)
impact.to_csv(INTERMEDIATE + "dupterms_metric_impact-r1.csv", index=False)
print(impact.to_string(index=False))

                           metric  genes_changed  pct_genes_changed  mean_before  mean_after  max_before  max_after  largest_single_drop
            gPS (unique diseases)            166               2.00       4.4488      4.4274         148        148                   -3
gps_TA (unique therapeutic areas)            167               2.02       2.4234      2.4146          20         20                   -1


### The band that matters: gps_TA ∈ [2, 5]

OR = 10.3 is defined on genes with a protein-altering variant and *intermediate* pleiotropy, operationalised
as 2–5 therapeutic areas. Genes crossing that boundary are the only ones whose classification actually
changes.

In [11]:
bands = []
for name, frame in [("gps_TA in [2,5]", ta), ("gPS in [2,5]", gps)]:
    b, a = frame["before"].between(2, 5), frame["after"].between(2, 5)
    bands.append(
        {
            "band": name,
            "genes_in_band_before": int(b.sum()),
            "genes_in_band_after": int(a.sum()),
            "net_change": int(a.sum() - b.sum()),
            "genes_leaving": int((b & ~a).sum()),
            "genes_entering": int((~b & a).sum()),
            "pct_of_band_reclassified": round(100 * (b != a).sum() / b.sum(), 2),
        }
    )
bands = pd.DataFrame(bands)
bands.to_csv(INTERMEDIATE + "dupterms_band_impact-r1.csv", index=False)
print(bands.to_string(index=False))

           band  genes_in_band_before  genes_in_band_after  net_change  genes_leaving  genes_entering  pct_of_band_reclassified
gps_TA in [2,5]                  4019                 3981         -38             52              14                      1.64
   gPS in [2,5]                  3504                 3504           0              8               8                      0.46


### Impact on the R3-mn-4 distribution table

In [12]:
before = pd.Series({t: primary_area(t) for t in QUAL_TERMS}).value_counts()
after = pd.Series({CANON_QUAL.get(t, t): primary_area(CANON_QUAL.get(t, t)) for t in QUAL_TERMS}).value_counts()
dist = pd.concat([before.rename("before"), after.rename("after")], axis=1).fillna(0).astype(int)
dist["delta"] = dist["after"] - dist["before"]
dist = dist.sort_values("before", ascending=False).rename_axis("therapeutic_area").reset_index()
dist.to_csv(INTERMEDIATE + "dupterms_distribution_effect-r1.csv", index=False)
print(dist.to_string(index=False))
print()
print(f"qualifying disease terms: {len(QUAL_TERMS)} -> {len({CANON_QUAL.get(t, t) for t in QUAL_TERMS})}")

                            therapeutic_area  before  after  delta
                                       other     574    542    -32
                      cancer or benign tumor     350    349     -1
                      nervous system disease     176    175     -1
                      cardiovascular disease     162    161     -1
                          infectious disease     143    142     -1
                   disorder of visual system     118    118      0
                       immune system disease     112    110     -2
musculoskeletal or connective tissue disease      97     96     -1
                    gastrointestinal disease      96     96      0
       reproductive system or breast disease      77     77      0
     injury, poisoning or other complication      65     65      0
                integumentary system disease      56     56      0
             respiratory or thoracic disease      49     49      0
                      urinary system disease      46     46   

**All but eight of the 40 collapsed terms come out of `other`** — 574 → 542. That is the same
conclusion from a different direction: duplication in our data is overwhelmingly a *phenotype-term*
phenomenon, and the `other` bucket is where the phenotype terms sit.

## How much is this an underestimate?

The deterministic rule is deliberately high-precision. A looser rule — any exact label or synonym string
shared across ontologies, no xref required — gives the other end of the range.

In [13]:
inverted = defaultdict(set)
for t in QUAL_TERMS:
    for s in SYN.get(t, ()):
        inverted[s].add(t)

loose = set()
for members in inverted.values():
    if len(members) < 2:
        continue
    for a, b in itertools.combinations(sorted(members), 2):
        if PREFIX[a] == PREFIX[b]:
            continue
        if b in ANC.get(a, ()) or a in ANC.get(b, ()):
            continue
        loose.add((a, b))

bound = pd.DataFrame(
    [
        {
            "rule": "deterministic (xref / xref+synonym)",
            "live_groups": len(LIVE_QUAL),
            "terms_affected": sum(len(m) for m in LIVE_QUAL.values()),
            "pct_of_2320": round(100 * sum(len(m) for m in LIVE_QUAL.values()) / len(QUAL_TERMS), 2),
        },
        {
            "rule": "loose (shared exact label/synonym only)",
            "live_groups": len(loose),
            "terms_affected": len({x for p in loose for x in p}),
            "pct_of_2320": round(100 * len({x for p in loose for x in p}) / len(QUAL_TERMS), 2),
        },
    ]
)
bound.to_csv(INTERMEDIATE + "dupterms_bounds-r1.csv", index=False)
print(bound.to_string(index=False))
print()
print("by ontology pair, loose rule:", dict(Counter("-".join(sorted([PREFIX[a], PREFIX[b]])) for a, b in loose)))
print()
print("the loose rule is NOT usable as an estimate -- it is visibly contaminated:")
for a, b in sorted(loose)[:14]:
    print(f"   {a:16s} {str(NAME.get(a))[:32]:34s} <-> {b:16s} {NAME.get(b)}")

                                   rule  live_groups  terms_affected  pct_of_2320
    deterministic (xref / xref+synonym)           40              80         3.45
loose (shared exact label/synonym only)           64             125         5.39

by ontology pair, loose rule: {'HP-MONDO': 27, 'EFO-HP': 22, 'EFO-MONDO': 14, 'DOID-MONDO': 1}

the loose rule is NOT usable as an estimate -- it is visibly contaminated:
   DOID_7551        gonorrhea                          <-> MONDO_0001056    gastric cancer
   EFO_0000195      metabolic syndrome                 <-> MONDO_0011565    metabolic syndrome X
   EFO_0000203      monoclonal gammopathy              <-> HP_0031047       Paraproteinemia
   EFO_0000266      aortic stenosis                    <-> MONDO_0042981    aortic valve stenosis
   EFO_0000274      atopic eczema                      <-> HP_0000964       Eczematoid dermatitis
   EFO_0000474      epilepsy                           <-> HP_0001250       Seizure
   EFO_0000693      sc

The loose rule doubles the count but pairs *gonorrhea* with *gastric cancer* (a shared "GC" synonym),
*epilepsy* with *Seizure* and *neurodegenerative disease* with *Brain atrophy*. It is an upper bound in the
literal sense only. **The honest range is 40 live groups (high precision) to ~64 (mostly false).**

## Precision of the 16 groups that touch gPS

Open Targets' own LLM benchmark on the same rule scored ~86% strictly equivalent, ~13% broader/narrower and
~0.5% wrong, with the **HP–MONDO class worst at ~38% broader/narrower**. Six of our 16 are HP–MONDO, so a
manual read of all 16 is cheap and worth having. Verdicts below are this notebook's judgement, not a
benchmark.

In [14]:
VERDICT = {
    "EFO_0004272": ("equivalent", "anemia (phenotype) vs anemia -- same entity, duplicated by term type"),
    "EFO_0000731": ("equivalent", "uterine fibroid = uterine leiomyoma"),
    "HP_0004398": ("equivalent", "peptic ulcer = peptic ulcer disease"),
    "HP_0000726": ("equivalent", "dementia, identical label across HP and MONDO"),
    "HP_0000175": ("equivalent", "cleft palate, identical label"),
    "EFO_0004994": (
        "equivalent",
        "lumbar disc degeneration vs intervertebral disk degeneration -- the issue's own example",
    ),
    "HP_0000099": ("equivalent", "glomerulonephritis, identical label"),
    "HP_0003077": ("equivalent", "hyperlipidemia, identical label"),
    "EFO_1001176": ("equivalent", "sensorineural hearing loss = sensorineural hearing impairment"),
    "HP_0001094": ("equivalent", "iridocyclitis, identical label"),
    "EFO_1000999": ("broader/narrower", "joint disease vs arthropathy -- near-synonymous, both very broad"),
    "EFO_0003086": ("broader/narrower", "kidney disease vs nephropathy -- near-synonymous, both very broad"),
    "EFO_0004280": ("broader/narrower", "movement disorder vs abnormality of movement -- broad phenotype class"),
    "EFO_0000706": ("different", "spondyloarthropathy (inflammatory) is not spondylosis (degenerative)"),
    "EFO_1000941": ("different", "frozen shoulder (adhesive capsulitis) is not bursitis"),
    "EFO_1001455": (
        "do not merge",
        "MONDO_0021205 IS the `disorder of ear` therapeutic-area root -- merging it would break the hierarchy",
    ),
}

live["verdict"] = [
    next((VERDICT[t][0] for t in row.split(" | ") for t in [t.split(" ")[0]] if t in VERDICT), "unreviewed")
    for row in live["terms"]
]
live["reason"] = [
    next((VERDICT[t][1] for t in row.split(" | ") for t in [t.split(" ")[0]] if t in VERDICT), "")
    for row in live["terms"]
]
live.to_csv(INTERMEDIATE + "dupterms_live_groups-r1.csv", index=False)

print(live["verdict"].value_counts().to_string())
print()
print(live[["terms", "studies", "verdict", "reason"]].to_string(index=False))

verdict
equivalent          10
broader/narrower     3
different            2
do not merge         1

                                                                                   terms  studies          verdict                                                                                               reason
                                  EFO_1000999 (joint disease) | HP_0003040 (Arthropathy)       76 broader/narrower                                     joint disease vs arthropathy -- near-synonymous, both very broad
                                 EFO_0003086 (kidney disease) | HP_0000112 (Nephropathy)       58 broader/narrower                                    kidney disease vs nephropathy -- near-synonymous, both very broad
                               EFO_0004272 (anemia (phenotype)) | MONDO_0002280 (anemia)       39       equivalent                                 anemia (phenotype) vs anemia -- same entity, duplicated by term type
                          EFO_00007

So of the 16 groups that reach gPS, **10 are safe merges, 3 are broad-but-arguable, 2 are wrong, and 1
must not be merged at all** because one member is a therapeutic-area root. Applying the merge to only the
10 safe ones would roughly halve every impact number above — which is worth stating, since the impact
figures reported here are therefore an **upper** bound on the true correction.

## Verdict

In [15]:
verdict = pd.DataFrame(
    [
        {
            "question": "Is our disease list contaminated by duplicated ontology terms?",
            "answer": "Yes, but marginally",
            "evidence": f"{len(LIVE_QUAL)} live groups / {sum(len(m) for m in LIVE_QUAL.values())} terms "
            f"of 2,320 qualifying disease terms ({summary.iloc[0]['pct_of_universe']}%)",
        },
        {
            "question": "Does it change gPS?",
            "answer": "No, not materially",
            "evidence": f"{int((gps['delta'] != 0).sum())} of 8,285 genes change; mean "
            f"{gps['before'].mean():.3f} -> {gps['after'].mean():.3f} "
            f"({100 * (gps['after'].mean() / gps['before'].mean() - 1):+.2f}%)",
        },
        {
            "question": "Does it change gps_TA?",
            "answer": "Marginally, and always downward",
            "evidence": f"{int((ta['delta'] != 0).sum())} of 8,285 genes change; mean "
            f"{ta['before'].mean():.3f} -> {ta['after'].mean():.3f}",
        },
        {
            "question": "Does it move genes across the OR=10.3 band?",
            "answer": "66 genes, 1.6% of the band",
            "evidence": f"{int(bands.iloc[0]['genes_in_band_before'])} -> "
            f"{int(bands.iloc[0]['genes_in_band_after'])} in gps_TA [2,5]; "
            f"{int(bands.iloc[0]['genes_leaving'])} leave, {int(bands.iloc[0]['genes_entering'])} enter",
        },
        {
            "question": "Is the damage independent of the `other` bucket problem?",
            "answer": "No -- they are the same problem",
            "evidence": f"{len(with_other)} of {len(live)} live gPS groups straddle a therapeutic-area boundary "
            f"with `other` on one side; 32 of the 40 collapsed terms come out of `other`",
        },
    ]
)
verdict.to_csv(INTERMEDIATE + "dupterms_verdict-r1.csv", index=False)
for r in verdict.itertuples():
    print(f"Q: {r.question}\n   -> {r.answer}\n      {r.evidence}\n")

Q: Is our disease list contaminated by duplicated ontology terms?
   -> Yes, but marginally
      40 live groups / 80 terms of 2,320 qualifying disease terms (1.72%)

Q: Does it change gPS?
   -> No, not materially
      166 of 8,285 genes change; mean 4.449 -> 4.427 (-0.48%)

Q: Does it change gps_TA?
   -> Marginally, and always downward
      167 of 8,285 genes change; mean 2.423 -> 2.415

Q: Does it move genes across the OR=10.3 band?
   -> 66 genes, 1.6% of the band
      4019 -> 3981 in gps_TA [2,5]; 52 leave, 14 enter

Q: Is the damage independent of the `other` bucket problem?
   -> No -- they are the same problem
      13 of 16 live gPS groups straddle a therapeutic-area boundary with `other` on one side; 32 of the 40 collapsed terms come out of `other`



### Reading

**Duplicated ontology terms are a real defect but a small one, and remapping them is not on the critical
path for this revision.** Merging every nominated group changes gPS for 2% of genes and moves its mean by
half a percent; it moves 1.6% of the genes in the band that defines OR = 10.3, in both directions.
Restricting to the 10 unambiguously correct merges would halve that again.

**What is worth saying to a referee is the interaction.** Thirteen of the sixteen duplicate groups that
reach gPS have the residual `other` bucket on one side, and 32 of the 40 collapsed qualifying terms come
out of `other`. The duplication only inflates the pleiotropy metric *because* the HP twin is unmappable to
a therapeutic area and `other` is counted as one. Excluding `other` from gps_TA — the fix already proposed
for the second problem, in `../ta-distribution/MAPPING_REVIEW.md` — neutralises most of this problem as a
side effect, without any remapping at all.

**Order of work implied:** fix `other`, re-measure, and only then decide whether the residual duplication
is worth a remap. It probably is not.

### Limitations

- Release **25.06 only**, deliberately. No ontology version change was made.
- The rule is high-precision and cross-ontology only; **same-ontology near-duplicates are invisible to
  it**, as are duplicate pairs with no shared xref and no shared exact synonym.
- The merge here is applied at the *term* level for counting purposes. A real remap would also have to
  reconcile study-level trait names, the credible-set tables and the ChEMBL indication join.
- The precision verdicts are this notebook's manual reading of 16 groups, not an independent benchmark.

## Two specific worries about what a remap would change

Both were raised as reasons a remap might be *necessary* rather than optional:

- **A** — remapping could deflate the apparent gains from non-European ancestries, if non-EUR studies sit
  on one twin and EUR studies on the other, so the non-EUR study gets credit for a "novel" gene that the
  EUR study had already found under the other term.
- **B** — remapping could increase the overlap with ChEMBL, because a GWAS on an HP term cannot match a
  ChEMBL indication in the MONDO tree, so genetic support is being lost.

Both are testable here, and both come out negative.

### Worry A — the ancestry result cannot be affected, structurally

Novelty in `../ancestry-vs-sample-size/01_discovery_regression.ipynb` is defined **per gene**:
`first_date = rows.groupby("geneId")["publicationDate"].transform("min")`, and a study is credited with a
novel gene when its publication date equals that gene's earliest date. **Disease terms appear nowhere in
that computation** — the notebook never reads `diseaseIds`. A gene first reported by any study on any trait
is not novel for anybody afterwards, whichever ontology term either study was mapped to.

The only route by which a remap could touch it is if a merged term moved a study between the disease and
measurement domains, since those are modelled separately. It cannot: no term in any live group is a
measurement term.

In [16]:
measurement_terms = set(onto.loc[onto.id == "EFO_0001444", "descendants"].iloc[0]) | {"EFO_0001444"}
live_terms = [t for members in LIVE_GPS.values() for t in members]
print(f"terms in live gPS duplicate groups: {len(live_terms)}")
print(f"  ... that are measurement terms: {sum(1 for t in live_terms if t in measurement_terms)}  <-- must be 0")
assert not any(t in measurement_terms for t in live_terms)

ancestry = pd.read_csv(INTERMEDIATE + "l2g_diseases_full-r1.csv", usecols=["studyId", "diseaseIds", "ancestryClass"])
ancestry["term"] = ancestry["diseaseIds"].map(ast.literal_eval)
ancestry = ancestry.explode("term").drop_duplicates(["studyId", "term"])

rows = []
for gi, members in LIVE_GPS.items():
    row = {"group": " | ".join(NAME.get(t, t) for t in members)}
    for k, t in enumerate(members):
        counts = ancestry.loc[ancestry["term"] == t, "ancestryClass"].value_counts().to_dict()
        row[f"side_{k + 1}"] = ", ".join(f"{a}={n}" for a, n in sorted(counts.items())) or "-"
    rows.append(row)
split = pd.DataFrame(rows)
split.to_csv(INTERMEDIATE + "dupterms_ancestry_split-r1.csv", index=False)
print()
print(split.to_string(index=False))

terms in live gPS duplicate groups: 32
  ... that are measurement terms: 0  <-- must be 0



                                                        group                    side_1                    side_2
                                          Dementia | dementia                 non-EUR=5 EUR=6, mixed=2, non-EUR=3
                                  anemia (phenotype) | anemia EUR=2, mixed=3, non-EUR=2 EUR=2, mixed=1, non-EUR=3
                      Glomerulonephritis | glomerulonephritis                 non-EUR=1                   mixed=1
                                Iridocyclitis | iridocyclitis                 non-EUR=3                 non-EUR=1
                              Hyperlipidemia | hyperlipidemia EUR=2, mixed=1, non-EUR=2          EUR=1, non-EUR=2
                          uterine fibroid | Uterine leiomyoma EUR=7, mixed=2, non-EUR=8            EUR=5, mixed=1
sensorineural hearing loss | Sensorineural hearing impairment                 non-EUR=1 EUR=1, mixed=2, non-EUR=1
                                 kidney disease | Nephropathy EUR=1, mixed=5, non-EUR=2

Even if novelty *were* defined per gene–disease pair, the mechanism would not fire: **ancestry classes
are mixed on both sides of essentially every twin**. There is no group where one term collects the European
studies and the other collects the non-European ones. The two smallest exceptions — Dementia (5 non-EUR)
versus dementia (6 EUR, 3 non-EUR, 2 mixed), and spondyloarthropathy (2 non-EUR) versus spondylosis
(3 mixed, 1 EUR) — are a handful of studies and the second pair is not a true duplicate anyway.

**Worry A is answered: the ancestry/novelty result is invariant to any remapping, by construction.**

### Worry B — does a remap recover lost ChEMBL matches?

The enrichment joins ChEMBL target–indication pairs against L2G evidence that has been **propagated up the
disease ontology**: evidence on term *D* counts for *D* and every ancestor of *D*, taking the maximum
score. The ChEMBL side is **not** propagated. So a match requires ChEMBL's indication to be *D* itself or
an ancestor of it — and an HP term's ancestors are all HP/phenotype terms, never MONDO. That is a real
asymmetry, and the worry is well founded a priori.

To measure it, the propagation is reimplemented in pandas and checked against the published parquet before
anything is changed.

In [17]:
l2g_enrich = pd.read_parquet(INTERMEDIATE + "l2g_full_for_enrichment", columns=["geneId", "score", "VEP", "diseaseIds"])
chembl = pd.read_parquet(INTERMEDIATE + "chembl_ti_pairs_maxphase-r1.parquet")
published_all = pd.read_parquet(INTERMEDIATE + "l2g_indirect_assoc_all-r1.parquet")
published_pav = pd.read_parquet(INTERMEDIATE + "l2g_indirect_assoc_pav-r1.parquet")

ONCOLOGY = set(onto.loc[onto.id == "MONDO_0045024", "descendants"].iloc[0]) | {"MONDO_0045024"}
print("oncology terms removed, as published:", len(ONCOLOGY))

# union semantics: coalescing two terms means the merged node inherits BOTH ancestor closures.
GROUP_OF = {}
enrich_universe = set(itertools.chain.from_iterable(l2g_enrich["diseaseIds"].map(list)))
for members in LIVE_GPS.values():
    present = [t for t in members if t in enrich_universe]
    if len(present) >= 2:
        for t in present:
            GROUP_OF[t] = present


def closure(term, merged):
    out = ANC.get(term, set()) | {term}
    if merged and term in GROUP_OF:
        for other in GROUP_OF[term]:
            out |= ANC.get(other, set()) | {other}
    return sorted(out - ONCOLOGY)


def propagate(frame, merged):
    """evidence_to_indirect_assosiations(use_max=True, efo_to_remove=oncology), in pandas."""
    ev = frame[["geneId", "score", "diseaseIds"]].copy()
    ev["diseaseId"] = ev["diseaseIds"].map(list)
    ev = ev.explode("diseaseId")
    ev = ev[~ev["diseaseId"].isin(ONCOLOGY)]
    ev["anc"] = ev["diseaseId"].map(lambda t: closure(t, merged))
    ev = ev.explode("anc").dropna(subset=["anc"])
    best = ev.groupby(["geneId", "anc"], as_index=False)["score"].max()
    return set(map(tuple, best[["geneId", "anc"]].values))


before_all = propagate(l2g_enrich, merged=False)
before_pav = propagate(l2g_enrich[l2g_enrich["VEP"] == 1], merged=False)
for label, mine, ref in [("all", before_all, published_all), ("PAV", before_pav, published_pav)]:
    theirs = set(map(tuple, ref[["targetId", "diseaseId"]].values))
    print(f"  {label}: reimplementation {len(mine)} pairs, published {len(theirs)} -- identical: {mine == theirs}")
    assert mine == theirs, "the pandas propagation must reproduce the published table exactly"

oncology terms removed, as published: 3593


  all: reimplementation 151704 pairs, published 151704 -- identical: True
  PAV: reimplementation 22509 pairs, published 22509 -- identical: True


In [18]:
after_all = propagate(l2g_enrich, merged=True)
after_pav = propagate(l2g_enrich[l2g_enrich["VEP"] == 1], merged=True)

ti_pairs = list(map(tuple, chembl[["targetId", "diseaseId"]].values))
APPROVED = (chembl["maxClinicalPhase"] >= 4).to_numpy().astype(int)

rows = []
for label, before, after in [("all evidence", before_all, after_all), ("PAV only", before_pav, after_pav)]:
    mb = np.array([p in before for p in ti_pairs])
    ma = np.array([p in after for p in ti_pairs])
    gained = ma & ~mb
    rows.append(
        {
            "definition": label,
            "chembl_ti_pairs": len(ti_pairs),
            "supported_before": int(mb.sum()),
            "supported_after": int(ma.sum()),
            "pairs_gained": int(gained.sum()),
            "pairs_lost": int((mb & ~ma).sum()),
            "approved_supported_before": int(APPROVED[mb].sum()),
            "approved_supported_after": int(APPROVED[ma].sum()),
            "approved_gained": int(APPROVED[gained].sum()),
            "nonapproved_gained": int((1 - APPROVED)[gained].sum()),
        }
    )
overlap = pd.DataFrame(rows)
overlap.to_csv(INTERMEDIATE + "dupterms_chembl_overlap-r1.csv", index=False)
print(overlap.to_string(index=False))
print()
gained_pairs = [
    (p, ph) for p, ph in zip(ti_pairs, chembl["maxClinicalPhase"]) if p in after_all and p not in before_all
]
print(f"every ChEMBL pair the remap recovers ({len(gained_pairs)}):")
for (t, disease), phase in sorted(gained_pairs, key=lambda x: -x[1]):
    print(f"   phase {phase}  {t}  {disease}  {NAME.get(disease, '?')}")

  definition  chembl_ti_pairs  supported_before  supported_after  pairs_gained  pairs_lost  approved_supported_before  approved_supported_after  approved_gained  nonapproved_gained
all evidence            37377               742              743             1           0                        242                       243                1                   0
    PAV only            37377               161              161             0           0                         72                        72                0                   0

every ChEMBL pair the remap recovers (1):
   phase 4.0  ENSG00000110148  HP_0004398  Peptic ulcer


**One pair out of 37,377.** The remap recovers a single target–indication pair —
`ENSG00000110148` (CCKBR) × `HP_0004398` peptic ulcer, an approved indication — and loses none. For the PAV
definition it recovers nothing at all.

The reason the asymmetry does not bite: ChEMBL's own indications are heavily EFO/MONDO but include 254 HP
terms, and our GWAS terms already propagate to their full ancestor closure. For a remap to gain a pair, a
ChEMBL indication has to sit *exactly* on the orphaned twin's side of the split. That almost never happens.

### What it does to OR = 10.3

In [19]:
genes_ta = pd.read_csv(INTERMEDIATE + "genes_therapeutic_areas.csv")


def or_rs(mask):
    m = np.asarray(mask, dtype=bool)
    n_support, n_none = int(m.sum()), int((~m).sum())
    x_support, x_none = int(APPROVED[m].sum()), int(APPROVED[~m].sum())
    from scipy.stats import fisher_exact

    odds, p = fisher_exact([[n_none - x_none, x_none], [n_support - x_support, x_support]])
    return {
        "n_supported_pairs": n_support,
        "n_approved": x_support,
        "OR": round(float(odds), 3),
        "RS": round((x_support / n_support) / (x_none / n_none), 3),
        "p": p,
    }


published_window = set(genes_ta.loc[genes_ta["uniqueTherapeuticAreas"].between(2, 5), "geneId"])
recomputed_window = set(ta.index[ta["after"].between(2, 5)])

scenarios = []
for label, support_set, window in [
    ("published (no remap)", before_pav, published_window),
    ("remap, published TA window", after_pav, published_window),
    ("remap + TA window recomputed after merge", after_pav, recomputed_window),
]:
    s = {(t, d) for t, d in support_set if t in window}
    scenarios.append({"scenario": label, "genes_in_window": len(window), **or_rs([p in s for p in ti_pairs])})
scenarios = pd.DataFrame(scenarios)
scenarios.to_csv(INTERMEDIATE + "dupterms_or_scenarios-r1.csv", index=False)
print(scenarios.to_string(index=False))

                                scenario  genes_in_window  n_supported_pairs  n_approved     OR    RS            p
                    published (no remap)             4028                 87          51 10.289 4.844 8.072448e-25
              remap, published TA window             4028                 87          51 10.289 4.844 8.072448e-25
remap + TA window recomputed after merge             3981                 93          52  9.212 4.620 6.091126e-24


The published OR reproduces exactly (**10.289**, 87 supported pairs, 51 approved), which validates the
whole chain. Under the remap with the published therapeutic-area window it is **unchanged**. If the window
is *also* recomputed after the merge — the internally consistent thing to do — it falls to **≈ 9.1**, and
that movement comes entirely from genes crossing the 2–5 boundary (defect measured earlier: 52 out, 14 in),
not from any change in ChEMBL overlap.

**Worry B is answered: remapping does not recover meaningful ChEMBL overlap.** The one thing that *does*
move the headline is the therapeutic-area window, and that is problem 2, not problem 1.

## Exports

| File | Contents |
| ---- | -------- |
| `dupterms_live_summary-r1.csv` | live duplicate groups per trait universe |
| `dupterms_live_groups-r1.csv` | every live gPS group: terms, study split, projects, therapeutic areas, manual verdict |
| `dupterms_metric_impact-r1.csv` | gPS and gps_TA before/after merging |
| `dupterms_band_impact-r1.csv` | movement across the gps_TA ∈ [2,5] band |
| `dupterms_distribution_effect-r1.csv` | effect on the R3-mn-4 therapeutic-area distribution |
| `dupterms_bounds-r1.csv` | deterministic versus loose rule, as a range |
| `dupterms_verdict-r1.csv` | the five questions and their answers |
| `dupterms_ancestry_split-r1.csv` | ancestry class of the studies on each side of every live twin |
| `dupterms_chembl_overlap-r1.csv` | ChEMBL target-indication pairs supported before and after the remap |
| `dupterms_or_scenarios-r1.csv` | OR = 10.3 under no remap, remap, and remap + recomputed window |